# Mirror-CFE: MNIST 7 vs. 9

Toy-Validierung der **Mirror-CFE**-Methode auf dem lokal trainierten **ResNet-18** (`mnist_7v9_resnet18.pth`).

Idee von Mirror-CFE: Der GAP-Latent `z` eines Bildes wird an der Entscheidungs-Hyperebene des Klassifikators **gespiegelt** (Paper Eq. 1), anschliessend per **L-BFGS** so verfeinert, dass der Klassifikator die Zielklasse ausgibt. Der gespiegelte Latent `z_r` wird mit einem **SimpleDecoder** (1x1-Features -> 28x28) zum Counterfactual-Bild dekodiert.

Unterschied zu den Fire/X-ray-Notebooks: Dort liegt ein custom-ResNet mit `model.encoder.blocks` / `model.decoder.decoder` vor. Hier ist es ein torchvision-ResNet-18 (`model.layer4` / `model.fc`) — die Mirror-Funktionen sind entsprechend angepasst. Statt des vollen Skip-Decoders (SSC/SPE/CSP) wird — wie fuer die MNIST-Toynotebooks ueblich — ein **SimpleDecoder** aus dem 1x1-GAP-Latent verwendet.

## 1. Imports & Konfiguration

In [ ]:
import ssl
from pathlib import Path

# macOS-Python bringt oft keine CA-Zertifikate mit -> MNIST-Download schlaegt
# mit CERTIFICATE_VERIFY_FAILED fehl. certifi-Bundle nutzen, sonst Pruefung lockern.
try:
    import certifi
    ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())
except ImportError:
    ssl._create_default_https_context = ssl._create_unverified_context

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import resnet18

torch.manual_seed(2024)
np.random.seed(2024)

DEVICE = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("Geraet:", DEVICE)

# ── Pfade (lokal) ─────────────────────────────────────────────────────────────
PROJ        = Path.cwd()
while PROJ.name and not (PROJ / "mnist_7v9_resnet18.pth").exists() and PROJ != PROJ.parent:
    PROJ = PROJ.parent
DATA_DIR    = PROJ / "data"
CLF_PATH    = PROJ / "mnist_7v9_resnet18.pth"
DECODER_OUT = PROJ / "Counterfactuals" / "Mirror" / "mirror_decoder_mnist.pth"

DIGITS      = (7, 9)
CLASS_NAMES = {0: "7", 1: "9"}      # 7 -> Klasse 0, 9 -> Klasse 1
MEAN, STD   = 0.1307, 0.3081

DEC_EPOCHS  = 10
BATCH_SIZE  = 128
print("Klassifikator vorhanden:", CLF_PATH.exists(), "->", CLF_PATH)

## 2. Daten — nur die Ziffern 7 und 9

Identische Vorverarbeitung wie beim Training des Klassifikators (7 -> 0, 9 -> 1).

In [ ]:
tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((MEAN,), (STD,)),
])

def make_loader(train, batch_size, shuffle=None):
    ds = datasets.MNIST(root=str(DATA_DIR), train=train, download=True, transform=tfm)
    idx = torch.where((ds.targets == DIGITS[0]) | (ds.targets == DIGITS[1]))[0]
    ds.targets = (ds.targets == DIGITS[1]).long()
    sh = train if shuffle is None else shuffle
    return DataLoader(Subset(ds, idx.tolist()), batch_size=batch_size,
                      shuffle=sh, num_workers=0)

train_loader = make_loader(True,  BATCH_SIZE)
test_loader  = make_loader(False, 64, shuffle=True)
print(f"Train-Bilder: {len(train_loader.dataset)} | Test-Bilder: {len(test_loader.dataset)}")

## 3. Klassifikator laden

Der ResNet-18 wird geladen und **eingefroren** (Mirror-CFE veraendert nur den Latent, nie das Modell).

In [ ]:
def build_classifier():
    m = resnet18(weights=None)
    # 1-Kanal-Stem, kein MaxPool (wie beim Training fuer 28x28)
    m.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    m.fc = nn.Linear(512, 2)
    return m

clf = build_classifier().to(DEVICE)
clf.load_state_dict(torch.load(CLF_PATH, map_location=DEVICE))
clf.eval()
for p in clf.parameters():
    p.requires_grad_(False)

# Test-Accuracy zur Kontrolle
correct = total = 0
with torch.no_grad():
    for x, y in test_loader:
        pred = clf(x.to(DEVICE)).argmax(1).cpu()
        correct += (pred == y).sum().item(); total += y.numel()
print(f"Klassifikator geladen & eingefroren | Test-Acc: {correct/total:.4f}")

## 4. Mirror-CFE Kernfunktionen

- **`extract_f4`** holt die letzte Feature-Map `(B,512,4,4)` per Forward-Hook auf `layer4`.
- **`get_boundary_params`** bildet die Mirror-Hyperebene `Wm = W[1]-W[0]`, `bm = b[1]-b[0]` (zeigt Richtung Klasse 1).
- **`position_function`** spiegelt den Latent (Paper Eq. 1): `z_r = z - 2k(Wm·z + bm)·Ŵm`.
- **`refine_with_lbfgs`** verfeinert `z_r`, bis `softmax(fc(z_r))` die getauschten (Ziel-)Wahrscheinlichkeiten trifft.

In [ ]:
def extract_f4(model, x):
    """Letzte Encoder-Feature-Map (B,512,4,4) per Hook auf layer4."""
    out = {}
    h = model.layer4.register_forward_hook(lambda m, i, o: out.update(f4=o))
    with torch.no_grad():
        _ = model(x)
    h.remove()
    return out["f4"]

def gap(f):
    return F.adaptive_avg_pool2d(f, 1).flatten(1)      # (B,512)

def get_boundary_params(model):
    W = model.fc.weight.data; b = model.fc.bias.data    # (2,512), (2,)
    return (W[1] - W[0]).clone(), (b[1] - b[0]).clone() # Richtung Klasse 1

def position_function(z, Wm, bm, k):
    """Paper Eq. 1: Spiegelung des Latents an der Entscheidungsgrenze."""
    W_hat = Wm / (Wm.norm() + 1e-8)
    dot   = (Wm.view(1, -1) * z).sum(1, keepdim=True) + bm
    return z - 2 * k * dot * W_hat.view(1, -1)

def predict_p1_from_z(model, z):
    """P(Klasse 1 = '9') direkt aus einem GAP-Latent z."""
    return torch.softmax(model.fc(z), 1)[:, 1]

def refine_with_lbfgs(model, z_init, src, tgt, orig_probs, iters=20):
    """L-BFGS: schiebt z, bis softmax(fc(z)) die getauschten Wahrscheinlichkeiten trifft."""
    swapped = orig_probs.clone().detach()
    idx = torch.arange(len(tgt))
    tmp = swapped[idx, tgt].clone()
    swapped[idx, tgt] = swapped[idx, src]; swapped[idx, src] = tmp
    z = Variable(z_init.clone().detach(), requires_grad=True)
    opt = torch.optim.LBFGS([z], lr=0.1)
    def closure():
        opt.zero_grad()
        loss = torch.norm(torch.softmax(model.fc(z), 1) - swapped) ** 2
        loss.backward(); return loss
    for _ in range(iters):
        opt.step(closure)
    return z.detach()

def compute_mirror_cfe(model, images, iters=20):
    """Vollstaendige Mirror-CFE im Latentraum. Rueckgabe: zr, zs, src, tgt, P(9)."""
    images = images.to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(model(images), 1)
        src   = probs.argmax(1); tgt = 1 - src
    zs = gap(extract_f4(model, images))
    Wm, bm = get_boundary_params(model); Wm, bm = Wm.to(DEVICE), bm.to(DEVICE)
    with torch.no_grad():
        zr_geo = position_function(zs.clone(), Wm, bm, k=1.0)
    zr = refine_with_lbfgs(model, zr_geo, src, tgt, probs.clone(), iters)
    return zr, zs, src, tgt, probs[:, 1]

print("Mirror-CFE Funktionen definiert")

## 5. Sanity-Check — Latent-Flip-Rate (ohne Decoder)

In [ ]:
imgs, lbls = next(iter(test_loader))
zr, zs, src, tgt, op = compute_mirror_cfe(clf, imgs)
cfe_pred  = (predict_p1_from_z(clf, zr) >= 0.5).long().cpu()
flip_rate = (cfe_pred == tgt.cpu()).float().mean().item()
print(f"Batch-Groesse        : {len(lbls)}")
print(f"Latent-Flip-Rate     : {flip_rate:.1%}  (Ziel: >80%)")
print(f"Quellklassen (Auszug): {[CLASS_NAMES[s] for s in src[:8].tolist()]}")
print(f"Zielklassen  (Auszug): {[CLASS_NAMES[t] for t in tgt[:8].tolist()]}")

## 6. SimpleDecoder — Definition & Training

Der Decoder lernt, aus dem 1x1-GAP-Latent `z` das normalisierte 28x28-Bild zu rekonstruieren. Zusaetzlich zum MSE haelt ein leichter **Klassifikator-Konsistenz-Term** die rekonstruierte Klasse stabil.

In [ ]:
class SimpleDecoder(nn.Module):
    """1x1-GAP-Latent (512) -> 28x28-Bild (normalisierter Raum)."""
    def __init__(self, in_dim=512):
        super().__init__()
        self.fc = nn.Linear(in_dim, 128 * 7 * 7)
        self.net = nn.Sequential(
            nn.BatchNorm2d(128), nn.ReLU(True),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),   # 7 -> 14
            nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),   # 14 -> 28
            nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, 1, 3, padding=1),
        )
    def forward(self, z):
        return self.net(self.fc(z).view(-1, 128, 7, 7))

dec = SimpleDecoder().to(DEVICE)
opt = torch.optim.Adam(dec.parameters(), lr=2e-3)

print("Decoder-Training...")
for ep in range(1, DEC_EPOCHS + 1):
    dec.train(); run = 0.0; n = 0
    for x, _ in train_loader:
        x = x.to(DEVICE)
        z = gap(extract_f4(clf, x))                 # eingefrorener Klassifikator-Latent
        rec = dec(z)
        with torch.no_grad():
            src_lbl = clf(x).argmax(1)
        loss = F.mse_loss(rec, x) + 0.1 * F.cross_entropy(clf(rec), src_lbl)
        opt.zero_grad(); loss.backward(); opt.step()
        run += loss.item() * x.size(0); n += x.size(0)
    print(f"  Ep {ep:02d}/{DEC_EPOCHS}: Loss {run/n:.4f}")
print("Decoder-Training abgeschlossen")

## 7. Decoder speichern -> `mirror_decoder_mnist.pth`

In [ ]:
torch.save({
    "model_state_dict": dec.state_dict(),
    "dataset": "mnist_7v9",
    "in_dim": 512,
    "epochs": DEC_EPOCHS,
}, DECODER_OUT)
print("SimpleDecoder gespeichert ->", DECODER_OUT)

## 8. Konfidenz-Journey & KFE-Uebergang (k = 0 -> 1)

Fuer einzelne Bilder wird der Latent linear von `z_s` (k=0, Original) nach `z_r` (k=1, Spiegelung) interpoliert. Pro Schritt wird (a) das dekodierte Bild gezeigt und (b) die Konfidenz `P(Zielklasse)` geplottet. Der erste Punkt mit `P(Ziel) >= 0.5` ist der **1. Counterfactual**.

In [ ]:
K_STEPS = [0.0, 0.25, 0.5, 0.6, 0.75, 1.0]

def denorm(t):
    return (t * STD + MEAN).clamp(0, 1)

def visualise_journey(model, decoder, images, labels, n_samples=4,
                      save_path=None):
    model.eval(); decoder.eval()
    images = images.to(DEVICE)
    n = min(n_samples, images.size(0))
    zr, zs, src, tgt, _ = compute_mirror_cfe(model, images[:n])

    # Konfidenz P(Zielklasse) pro k
    p_tgt = []   # (len(K), n)
    decoded = []  # (len(K), n, 1, 28, 28)
    with torch.no_grad():
        for k in K_STEPS:
            zk = (1 - k) * zs + k * zr
            probs = torch.softmax(model.fc(zk), 1)
            p_tgt.append(probs[torch.arange(n), tgt].cpu().numpy())
            decoded.append(decoder(zk).cpu())

    n_cols = 1 + len(K_STEPS) + 1
    fig = plt.figure(figsize=(n_cols * 1.9, n * 2.3))
    gs  = gridspec.GridSpec(n, n_cols, figure=fig, hspace=0.55, wspace=0.25)
    k_lbl = ["k=0\n(Original)", "k=0.25", "k=0.5\n(Grenze)",
             "k=0.6", "k=0.75", "k=1.0\n(Spiegel)"]

    for i in range(n):
        s, t, tr = src[i].item(), tgt[i].item(), int(labels[i])
        # Original
        ax = fig.add_subplot(gs[i, 0])
        ax.imshow(denorm(images[i,0].cpu()), cmap="gray"); ax.axis("off")
        col = "green" if s == tr else "red"
        ax.set_title(f"Original\nwahr {CLASS_NAMES[tr]} / pred {CLASS_NAMES[s]}",
                     fontsize=7, color=col)
        # KFE-Schritte: dekodierte Bilder
        for j, k in enumerate(K_STEPS):
            ax = fig.add_subplot(gs[i, j + 1])
            ax.imshow(denorm(decoded[j][i,0]), cmap="gray"); ax.axis("off")
            p = p_tgt[j][i]
            ax.set_title(f"{k_lbl[j]}\nP({CLASS_NAMES[t]})={p:.2f}", fontsize=6.5)
            if abs(k - 0.5) < 1e-6:
                for sp in ax.spines.values(): sp.set_edgecolor("cyan"); sp.set_linewidth(2)
        # Journey-Plot
        axc = fig.add_subplot(gs[i, -1])
        confs = [p_tgt[j][i] for j in range(len(K_STEPS))]
        axc.plot(K_STEPS, confs, "o-", color="steelblue", lw=2, ms=4)
        axc.axhline(0.5, color="k", ls="--", lw=1)
        for k, p in zip(K_STEPS, confs):
            if p >= 0.5:
                axc.scatter([k], [p], color="limegreen", s=60, zorder=5); break
        axc.set_ylim(-0.05, 1.05); axc.set_xlim(-0.05, 1.05)
        axc.set_xlabel("k", fontsize=7); axc.tick_params(labelsize=6)
        flipped = confs[-1] >= 0.5 or max(confs) >= 0.5
        axc.set_title(f"{CLASS_NAMES[s]} -> {CLASS_NAMES[t]}\n"
                      f"{'gekippt' if flipped else 'nicht gekippt'}",
                      fontsize=7, color="green" if flipped else "red")

    plt.suptitle("Mirror-CFE — KFE-Uebergang & Konfidenz-Journey (cyan = Grenze k=0.5)",
                 y=1.01, fontsize=11)
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.show()

visualise_journey(clf, dec, imgs, lbls, n_samples=4,
                  save_path=str(PROJ / "Counterfactuals" / "Mirror" / "mirror_cfe_mnist_journey.png"))

## 9. Counterfactual-Bilder: Original | CFE | Differenz

Der gespiegelte Latent `z_r` (k=1) wird dekodiert. Das Differenzbild `|x' - x|` zeigt, **welche Pixel** die Methode aendert, um 7 <-> 9 zu kippen.

In [ ]:
def visualise_cfe_images(model, decoder, images, labels, n_samples=6, save_path=None):
    model.eval(); decoder.eval()
    images = images.to(DEVICE)
    n = min(n_samples, images.size(0))
    zr, zs, src, tgt, _ = compute_mirror_cfe(model, images[:n])
    with torch.no_grad():
        cfe   = decoder(zr)
        rec   = decoder(zs)
        preds = model(cfe).argmax(1).cpu()

    fig, axes = plt.subplots(n, 4, figsize=(9, n * 2.2))
    if n == 1: axes = axes[None, :]
    titles = ["Original x", "Rekon (k=0)", "CFE x' (k=1)", "|x' - x|"]
    for c, t_ in enumerate(titles):
        axes[0, c].set_title(t_, fontsize=10, fontweight="bold")
    for i in range(n):
        o = denorm(images[i,0].cpu()).numpy()
        r = denorm(rec[i,0].cpu()).numpy()
        c = denorm(cfe[i,0].cpu()).numpy()
        d = np.abs(c - o)
        axes[i,0].imshow(o, cmap="gray")
        axes[i,1].imshow(r, cmap="gray")
        axes[i,2].imshow(c, cmap="gray")
        axes[i,3].imshow(d, cmap="hot")
        for j in range(4): axes[i,j].axis("off")
        flipped = preds[i].item() == tgt[i].item()
        col = "green" if flipped else "red"
        axes[i,2].set_xlabel(f"-> {CLASS_NAMES[tgt[i].item()]} "
                             f"({'ok' if flipped else 'x'})", fontsize=8, color=col)
        axes[i,2].axis("on"); axes[i,2].set_xticks([]); axes[i,2].set_yticks([])
        axes[i,0].set_ylabel(CLASS_NAMES[src[i].item()], fontsize=10, rotation=0,
                             labelpad=12, va="center"); axes[i,0].axis("on")
        axes[i,0].set_xticks([]); axes[i,0].set_yticks([])
    plt.suptitle("Mirror-CFE + SimpleDecoder: Original | Rekon | CFE | Differenz",
                 y=1.005, fontsize=11)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.show()

visualise_cfe_images(clf, dec, imgs, lbls, n_samples=6,
                     save_path=str(PROJ / "Counterfactuals" / "Mirror" / "mirror_cfe_mnist_images.png"))

## 10. Batch-Evaluation — Flip-Raten

- **Latent-Flip-Rate**: Anteil, bei dem der gespiegelte Latent `z_r` die Zielklasse erreicht (Kern der Methode).
- **Decodierte Flip-Rate**: Anteil, bei dem das **dekodierte** CFE-Bild vom Klassifikator als Zielklasse erkannt wird.

In [ ]:
n_batches = 10
lat_flip = dec_flip = total = 0
for bx, by in list(test_loader)[:n_batches]:
    zr, zs, src, tgt, _ = compute_mirror_cfe(clf, bx)
    lat_pred = (predict_p1_from_z(clf, zr) >= 0.5).long().cpu()
    with torch.no_grad():
        dec_pred = clf(dec(zr)).argmax(1).cpu()
    lat_flip += (lat_pred == tgt.cpu()).sum().item()
    dec_flip += (dec_pred == tgt.cpu()).sum().item()
    total    += len(by)

print("=" * 50)
print("Mirror-CFE — MNIST 7 vs. 9")
print("=" * 50)
print(f"  Bilder evaluiert       : {total}")
print(f"  Latent-Flip-Rate       : {lat_flip/total:.1%}")
print(f"  Decodierte Flip-Rate   : {dec_flip/total:.1%}")
print("=" * 50)